# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,30.5,0.027681,-0.593333,0.203333,25,6
1,CTR vs PD,PES_RG,25.5,0.014256,-0.660000,0.170000,25,6
2,CTR vs PD,PES_BY,31.0,0.029597,-0.586667,0.206667,25,6
3,CTR vs PD,VKS_MajRad,32.0,0.030893,-0.573333,0.213333,25,6
4,CTR vs PD,VKS_MinRad,30.0,0.023146,-0.600000,0.200000,25,6
5,CTR vs PD,VKS_Angle,71.0,0.864670,-0.053333,0.473333,25,6
6,HC vs protan,TES,5.0,0.000029,-0.955556,0.022222,25,9
7,HC vs protan,PES_RG,0.0,0.000012,-1.000000,0.000000,25,9
8,HC vs protan,PES_BY,50.0,0.015495,-0.555556,0.222222,25,9
9,HC vs protan,VKS_MajRad,6.0,0.000035,-0.946667,0.026667,25,9


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,7.0,0.000006,-0.930000,0.035000,25,8
15,HC vs deutan,VKS_MajRad,8.0,0.000010,-0.920000,0.040000,25,8
7,HC vs protan,PES_RG,0.0,0.000012,-1.000000,0.000000,25,9
13,HC vs deutan,PES_RG,0.0,0.000029,-1.000000,0.000000,25,8
6,HC vs protan,TES,5.0,0.000029,-0.955556,0.022222,25,9
9,HC vs protan,VKS_MajRad,6.0,0.000035,-0.946667,0.026667,25,9
12,HC vs deutan,TES,7.5,0.000110,-0.925000,0.037500,25,8
10,HC vs protan,VKS_MinRad,15.0,0.000153,-0.866667,0.066667,25,9
14,HC vs deutan,PES_BY,22.0,0.001129,-0.780000,0.110000,25,8
11,HC vs protan,VKS_Angle,185.0,0.004945,0.644444,0.822222,25,9


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
16,HC vs deutan,VKS_MinRad,7.0,0.000006,-0.930000,0.035000,25,8,0.000039,True
15,HC vs deutan,VKS_MajRad,8.0,0.000010,-0.920000,0.040000,25,8,0.000048,True
7,HC vs protan,PES_RG,0.0,0.000012,-1.000000,0.000000,25,9,0.000074,True
13,HC vs deutan,PES_RG,0.0,0.000029,-1.000000,0.000000,25,8,0.000116,True
9,HC vs protan,VKS_MajRad,6.0,0.000035,-0.946667,0.026667,25,9,0.000146,True
6,HC vs protan,TES,5.0,0.000029,-0.955556,0.022222,25,9,0.000146,True
12,HC vs deutan,TES,7.5,0.000110,-0.925000,0.037500,25,8,0.000330,True
10,HC vs protan,VKS_MinRad,15.0,0.000153,-0.866667,0.066667,25,9,0.000458,True
14,HC vs deutan,PES_BY,22.0,0.001129,-0.780000,0.110000,25,8,0.002258,True
11,HC vs protan,VKS_Angle,185.0,0.004945,0.644444,0.822222,25,9,0.009890,True
